### Anotações RAG LLM

In [34]:
# Retrieval: A grande questão dessa parte é que um dado prompt irá buscar um contexto específico numa vectorstore
# para ser respondido, a resposta será Aumentada com os dados obtidos nessa vectorstore. Por exemplo: Uma pergunta específica
# é feita sobre um item numa db. O LLM precisa gerar um contexto q será embeddado e irá buscar a informação por similaridade vetorial
# na vectorstore, recuperar essa informação, transformá-la em chunks e adicionar ao contexto do output

# Trade-off no tamanho dos chunks: Quando estamos chunkenizando um texto, estamos dividindo-o em pedaços menores que sejam 
# interpretáveis por um LLM. Ele deve ser pequeno o bastante para possibilitar mais combinações possíveis entre diferentes
# chunks e melhorar a qualidade das respostas, mas precisa ser grande o suficiente para que aquele mesmo bloco contenha
# significado o suficiente para que seja entendido dentro de um contexto maior

# Embedding: uma vez que temos um chunk, um embedding é a representação vetorial dos chunks. Isso é feito através dos transformers
# O chunking em si é classificado como um processo pré-transformer

# Document Loader: Uma abstração que nos permite pensar numa maneira estruturada de inputar diferentes documentos, de diferentes 
# formatos, de diferentes fontes. Um loader é um metodo de carregar os dados de documentos diversos e temos diversos tipos de loaders dispo
# niveis na comunidade langchain

# Prompt template: é a estrutura geral de um prompt, que recebe como input um dicionário
# q sao os valores q iremos usar para preencher nosso prompt.
# Quando usamos na prática, a clase prompt template recebe input variable, q é um dict de variáveis
# q irão popular nosso template e recebe o template em si

# Chain: Chain é um sequenciamento de LLMs, que usam o output de 1 como input do próximo,
# possibilitando coisas como multimodalidade, niveis de abstração e raciocinio superiores,
# desse modo, podemos criar uma unica aplicação coerente

# Output parser: é uma função responsável por capturar o output de um modelo de linguagem e parseá-lo
# em um formato mais utilizável, como um dict ou json, de forma a garantir que consiga ser usada no fluxo
# abaixo (como input da proxima camada do modelo por exemplo). é importante vc fornecer schemas para o output
# parser, para que ele retorne no formato adequado, isso entra como o parametro partial_variables na classe
# Prompt Template (isso é chamado de objeto pydantic)

# Agentes: Podemos pensar nos agentes como robos autonomos. Eles recebem uma instrucao e tentam quebrar
# aquilo, através de sua racionalidade, em subtasks, buscam ferramentas de LLM necessárias pra isso, fazem 
# chamadas API ou scrapam a internet para buscar o resultado que querem. A racionalizacao do o1 é um exemplo disso
# Uma forma de pensar em agentes é o seguinte: nem sempre a chain q vc define é estática, as vezes, dependendo do 
# tipo de prompt, um agente pode modelar uma chain customizada on the go para satistazer o usuario.
# Um agente tem um conjunto de tools para construir uma custom chain q é dependente do prompt do user

# Uma tool é a ferramenta que o agente usa para executar oq precisa, como por exemplo suas funcoes e instrucoes
# de como proceder (equivalente a uma função do python). um exemplo de tool é um webscraper simples por exemplo

# Um react agent é um tipo de abordagem que representa reasoning + action, ou seja. Consegue raciocionar sobre informa
# coes de maneira sequencial de modo a conseguir escolher melhores as ações q vai realizar. É uma mistura de chain of thought
# que define um raciocinio que precede a ação

# o agent exectutor é o componente orquestrador de toda a ação do LLM, ele coordena como o agente decide cada novo passo que é dado
# como eles sao executados e quando se chega numa conclusao satisfatoria pra parar. ele permite fornecer ciclos de feedback, de modo
# a modificar os prompts iniciais para que cheguem mais proximos da saida desejada e parem quando necessario (stop sequence)
# prompt hub: um repositorio publico de prompts ja pre construidos e definidos pela comunidade (por exemplo do langchain) q já foram
# validados pela comunidade como mais eficientes para atingir objetivos. podem seri importados diretamente no codigo https://smith.langchain.com/hub

# agent executor, ele vai ser o orquestrador de tudo e invocador das funcoes

# modelo pydantic. O modelo pydantic é um em que o desenvolvedor pode definir uma estrutura Json que seria o "alvo" do output parser.
# ou seja, é definida a maneira que o desenvolvedor gostaria de serializar um determinado output de texto, e entao o LLM busca criar um
# "match" entre o formato desejado e o conteúdo do texto

# LCEL, langchain expression language, é uma maneira declarativa de encadear chains juntas de maneira mais eficiente, permitindo assincronidade
# batch, stremaing, paralelismo e integracao com ferramentas como langsmith

# entender oq é um callback

### Estrutura da POC S.I

In [35]:
# DUC (Dividas Ativas) https://capital.sp.gov.br/web/fazenda/w/servicos/27233
# Fotos Earth/Maps/StreetView

# Fazer PDF bonito, estético, com 5 bullets no começo

# Dados Futuros:
# Valor Venal do Imível
# Licença Ambiental (Cetesb)
# APIs SAEC (certidoes)
# API Jusbrasil (processos judiciais)
# Matricula Imovel Cartorios

### Importa bibliotecas

In [36]:
import openai
import re
import sys
import os
import numpy        as np
import faiss
import io
from io import BytesIO

import pandas            as pd
import geopandas         as gpd
import matplotlib.pyplot as plt
import contextily        as ctx
from   shapely.wkt       import loads

from reportlab.lib.pagesizes       import A4
from reportlab.lib.styles          import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus            import SimpleDocTemplate, Paragraph, Spacer, Image
from reportlab.lib                 import colors
from PIL                           import Image as PILImage

from reportlab.pdfbase.ttfonts     import TTFont
from reportlab.pdfbase             import pdfmetrics

### Bibliotecas próprias

In [37]:
sys.path.append(r'D:\Google Drive\Meu Drive\Codigos bee6\bee6_notebooks\bee6_geral')
from importacao_dados import importa_parquet, importa_pdfs_texto

# Configure a chave da API do OpenAI
os.environ["OPENAI_API_KEY"] = "OPENAI_API_KEY_REMOVED"

### Arquivos próprios

In [38]:
# Importa os arquivos parque e faz as definições dos dataframes
dfs_parquet = importa_parquet(r"D:\Google Drive\Meu Drive\Codigos bee6\bee6_arquivos\solo_inteligente\Tratados")
lotes_iptu = dfs_parquet['lotes_iptu']
lei16 = dfs_parquet['lei16']

### Prepara os conjuntos de dados para serem livremente buscados

In [39]:
# Gera o forward fill para deixar todos os SQL com uma geometry (os vários apartamentos de mesmo logradouro, por exemplo)
lotes_iptu['numero do imovel'] = lotes_iptu['numero do imovel'].fillna(0).replace([np.inf, -np.inf], 0).astype(int).astype(str)
lotes_iptu['end'] = lotes_iptu['nome de logradouro do imovel'] + ', ' + lotes_iptu['numero do imovel']
lotes_iptu['geometry'] = lotes_iptu['geometry'].ffill()

lei16 = gpd.GeoDataFrame(
    lei16.assign(geometry_lei16=lei16['geometry'].apply(loads)), geometry='geometry_lei16', crs="EPSG:32723")
lei16 = lei16.drop('geometry', axis = 1)

### Faz a pesquisa por lote

In [40]:
lista_sqls = ['3101070032','1201940003','1722000031','1201610005']

# Usando f-string (forma 1)
pesquisa_sql = lotes_iptu.query(f"sql == '{lista_sqls[3]}'")

pesquisa_sql = gpd.GeoDataFrame(
    pesquisa_sql.assign(geometry=pesquisa_sql['geometry'].apply(loads)),geometry='geometry',crs="EPSG:32723")

### Busca o polígono de zoneamento mais próximo para um dado SQL

In [41]:
def atualiza_pesquisa_com_lei16_maior_intersec(pesquisa_sql, lei16):
    lei16 = lei16.rename(columns={'geometry_lei16': 'geometry'}).set_geometry('geometry')
    pesquisa_sql = pesquisa_sql.set_geometry('geometry')
    
    # Verificar e alinhar o CRS
    if pesquisa_sql.crs != lei16.crs:
        lei16 = lei16.to_crs(pesquisa_sql.crs)
    
    # Realiza o sjoin - mantém a geometria do pesquisa_sql
    merged = gpd.sjoin(pesquisa_sql, lei16, how='left', predicate='intersects')
    
    # Se não houver correspondências, retorna pesquisa_sql sem alterações
    if merged.empty:
        return pesquisa_sql
    
    # Agora precisamos recuperar a geometria do lei16 usando o index_right
    # index_right mapeia qual registro de lei16 foi associado
    merged = merged.merge(
        lei16[['geometry']],left_on='index_right', 
        right_index=True, how='left', suffixes=('', '_lei16'))
    
    # Calcula a área de interseção
    merged['intersect_area'] = merged.apply(
        lambda row: row.geometry.intersection(row.geometry_lei16).area if row.geometry_lei16 is not None else 0,
        axis=1)
    
    # Seleciona a linha com a maior área de interseção
    if merged['intersect_area'].max() == 0:
        # Nenhuma interseção real, então retorna pesquisa_sql
        return pesquisa_sql
    
    idx_max = merged['intersect_area'].idxmax()
    final = merged.loc[[idx_max]]
    
    # Garantir que retorne apenas a primeira linha (apesar de já ser apenas uma)
    final = final.head(1)
    
    return final

pesquisa_sql = atualiza_pesquisa_com_lei16_maior_intersec(pesquisa_sql, lei16)

# Extrai o SQL para nomear o pdf e a imagem
valor_sql = pesquisa_sql["sql"].iloc[0]  # Ajuste conforme a necessidade

valor_sql_limpo = re.sub(r'[^\w\s-]', '', str(valor_sql))
valor_sql_limpo = re.sub(r'\s+', '_', valor_sql_limpo).strip()

### A partir dos dados do Imovel pesquisado, retorna uma string com todo o conteudo pesquisado nos arquivos Faiss (RAG)

In [42]:
##############################################################################
#                              VARIÁVEIS GLOBAIS                             #
##############################################################################

# 1) Mensagem de sistema para o GPT-4 (TUDO concentrado aqui)
SYSTEM_MESSAGE = (
    "Você é um consultor avançado de regularização fundiária da empresa Solo Inteligente. "
    "Sua função é produzir um único texto final formatado em português, contendo: \n\n"
    "• **Três bullet points iniciais** dando uma visão rápida dos pontos mais importantes abordados no relatório.\n\n"
    "1) Um título: 'Relatório de Regularização Solo Inteligente'.\n"
    "2) Dados gerais do imóvel a partir do DataFrame 'pesquisa_sql' (ex.: número SQL, frente, "
    "área do terreno, área edificada, matrícula, codlog, etc.).\n"
    "3) Dados gerais do zoneamento (ex.: zona de uso, frente mínima, taxa de ocupação, coeficientes, etc.).\n"
    "4) Uma seção de 'Conclusões' explicando as características do imóvel e como elas "
    "se relacionam com a legislação municipal consultada nos documentos .faiss, de acordo com a zona em que se encontra.\n"
    "5) Identifique possíveis inconsistências ou divergências entre as características do imóvel e "
    "a legislação aplicada à zona, referenciando quando necessário as normas encontradas.\n"
    "6) Proponha soluções ou caminhos para resolver essas inconsistências, visando a regularização.\n\n"
    "Use linguagem profissional e objetiva; não inclua links externos, mas cite leis ou decretos relevantes "
    "quando necessário. Fundamente-se sempre nos 'trechos recuperados' do .faiss. Destaque pontos críticos "
    "ou divergências de forma clara e encerre com recomendações concretas para o proprietário.\n\n"
    "Cada resposta deve ser coesa, sem repetições desnecessárias, e pronta para ser convertida em PDF.\n"
)

# 2) Caminho do índice FAISS contendo a documentação jurídica
VECTOR_DB_PATH = r"D:\Google Drive\Meu Drive\Codigos bee6\bee6_arquivos\solo_inteligente\Vector Dbs\docs_prefeitura.faiss"

# 3) Chave de API do OpenAI (leia de variável de ambiente ou defina aqui se preferir)
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

##############################################################################
#                                 CLASSE BUSCARAG                            #
##############################################################################

class BuscaRAG:
    def __init__(self, api_key, system_message):
        self.api_key = api_key
        self.system_message = system_message

    def carregar_banco_vetorial(self, caminho_db):
        if not os.path.exists(caminho_db):
            raise FileNotFoundError(f"Banco vetorial não encontrado no caminho: {caminho_db}")
        return faiss.read_index(caminho_db)

    def gerar_embedding(self, texto):
        openai.api_key = self.api_key
        response = openai.Embedding.create(
            input=texto,
            model="text-embedding-ada-002"
        )
        return np.array(response["data"][0]["embedding"]).astype("float32")

    @staticmethod
    def criar_consulta(dados_linha):
        """
        Constrói a 'consulta' textual a partir dos dados do imóvel
        presentes em uma linha do DataFrame (pesquisa_sql).
        """
        conteudo = ", ".join(
            f"{col}: {val}"
            for col, val in dados_linha.items()
            if pd.notna(val)
        )
        return f"Informações do imóvel e zoneamento: {conteudo}."

    @staticmethod
    def buscar_no_banco_vetorial(indice, embedding_consulta, k=5):
        """
        Realiza a busca vetorial no índice FAISS e retorna
        as distâncias e índices dos k resultados mais relevantes.
        """
        distancias, indices = indice.search(embedding_consulta.reshape(1, -1), k)
        return distancias[0], indices[0]

    def enriquecer_consulta(self, consulta, textos_recuperados):
        """
        Monta o prompt final (mensagem do usuário) e executa a chamada
        ao modelo GPT-4, passando o SYSTEM_MESSAGE e o prompt construído.
        """
        openai.api_key = self.api_key

        user_prompt = (
            f"{consulta}\n\n"
            f"Estes são os trechos recuperados do banco vetorial:\n"
            f"{textos_recuperados}\n"
        )

        response = openai.ChatCompletion.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": self.system_message},
                {"role": "user", "content": user_prompt}
            ]
        )
        return response["choices"][0]["message"]["content"]

    def enriquecer_relatorio_com_rag(self, dados_do_imovel, indice, k):
        """
        Gera a 'consulta', obtém os trechos semelhantes no .faiss,
        e pede ao GPT-4 para produzir o texto final de relatório.
        """
        consulta = self.criar_consulta(dados_do_imovel)
        embedding_consulta = self.gerar_embedding(consulta)

        distancias, indices_retornados = self.buscar_no_banco_vetorial(
            indice, embedding_consulta, k
        )

        # Aqui, apenas indicamos os índices retornados. 
        textos_recuperados_list = [
            f"Trecho relevante do documento com índice {idx}"
            for idx in indices_retornados
        ]
        textos_recuperados = "\n".join(textos_recuperados_list)

        # Chama a função que efetivamente consulta o modelo GPT-4
        texto_final = self.enriquecer_consulta(consulta, textos_recuperados)
        return texto_final

    def executar_busca_rag(self, pesquisa_sql, vector_db_path, k=3):
        """
        1) Carrega o banco vetorial,
        2) Para cada imóvel no DataFrame 'pesquisa_sql', gera a consulta + busca RAG
        3) Retorna um único texto final (ou vários, caso o DF tenha várias linhas).
        """
        indice = self.carregar_banco_vetorial(vector_db_path)
        relatorios = []

        for i, linha in pesquisa_sql.iterrows():
            texto_relatorio = self.enriquecer_relatorio_com_rag(
                dados_do_imovel=linha, indice=indice, k=k
            )
            relatorios.append(f"### Relatório do Imóvel {i+1} ###\n{texto_relatorio}\n")

        return "\n".join(relatorios)

##############################################################################
#                         FUNÇÃO PRINCIPAL PARA USO                          #
##############################################################################

def gerar_relatorio_final(pesquisa_sql):
    """
    Função de alto nível que:
    1) Instancia a classe BuscaRAG;
    2) Executa a busca RAG;
    3) Retorna a string final do relatório consolidado.
    """
    busca_rag = BuscaRAG(
        api_key=OPENAI_API_KEY,
        system_message=SYSTEM_MESSAGE
    )

    string_relatorio = busca_rag.executar_busca_rag(
        pesquisa_sql=pesquisa_sql,
        vector_db_path=VECTOR_DB_PATH,
        k=3
    )
    return string_relatorio

##############################################################################
#                          APLICAÇÃO DA FUNÇÃO                               
##############################################################################

# Aqui presumimos que a variável 'pesquisa_sql' já está definida globalmente
# e contém os dados do(s) imóvel(is) a serem analisados.

string_relatorio = gerar_relatorio_final(pesquisa_sql)

### Função: Plota a planta de situação com layer do open street maps

In [43]:
def plotar_planta_situacao(pesquisa_sql, valor_sql_limpo, 
                           crs_original="EPSG:32723", 
                           crs_map="EPSG:3857", 
                           dpi=1000, 
                           zoom_out_factor=2.7):
    """
    Gera um plot baseado na geometria do lote em 'pesquisa_sql', com mais zoom out para mais contexto,
    e salva no diretório especificado com nome personalizado. Também retorna a imagem em buffer.

    Args:
        pesquisa_sql (pd.DataFrame): DataFrame com a coluna 'geometry'.
        valor_sql_limpo (str): Valor para compor o nome do arquivo.
        crs_original (str): CRS original dos dados (ex: "EPSG:32723").
        crs_map (str): CRS para o mapa base (ex: "EPSG:3857").
        dpi (int): Resolução da imagem em pontos por polegada.
        zoom_out_factor (float): Fator para aumentar os limites da visualização (porcentagem).

    Returns:
        str: Caminho completo do arquivo salvo.
        io.BytesIO: Buffer contendo a imagem do plot em formato PNG.
    """
    # Diretório de destino
    diretorio = r"D:\Google Drive\Meu Drive\Codigos bee6\bee6_arquivos\solo_inteligente\Imagens Planta"

    # Garantir que o diretório existe
    os.makedirs(diretorio, exist_ok=True)

    # Nome do arquivo
    nome_arquivo = f"imagem_lote_{valor_sql_limpo}.png"
    caminho_completo = os.path.join(diretorio, nome_arquivo)

    # Extrair a geometria do primeiro registro (apenas o lote)
    lote_geometry = pesquisa_sql.iloc[0]['geometry']

    # Criar um GeoDataFrame para o lote
    gdf_lote = gpd.GeoDataFrame({'geometry': [lote_geometry]}, crs=crs_original)

    # Converter para o CRS do mapa (ex: EPSG:3857), usado pelo contextily
    gdf_lote = gdf_lote.to_crs(crs_map)

    # Criar a figura e eixos para o plot
    fig, ax = plt.subplots(figsize=(12, 12))

    # Plotar o lote em vermelho
    gdf_lote.plot(ax=ax, color='red', edgecolor='black', alpha=0.7, label='Lote')

    # Ajustar o aspecto do gráfico para não distorcer as geometrias
    ax.set_aspect('equal', 'box')

    # Expandir os limites do plot para adicionar zoom out
    xmin, ymin, xmax, ymax = gdf_lote.total_bounds
    x_range = xmax - xmin
    y_range = ymax - ymin
    ax.set_xlim(xmin - zoom_out_factor * x_range, xmax + zoom_out_factor * x_range)
    ax.set_ylim(ymin - zoom_out_factor * y_range, ymax + zoom_out_factor * y_range)

    # Adicionar o mapa base do OpenStreetMap
    ctx.add_basemap(ax, crs=gdf_lote.crs, source=ctx.providers.OpenStreetMap.Mapnik)

    # Configurar título e legenda
    ax.set_title('Planta de Situação', fontsize=16)
    ax.legend()
    ax.set_xlabel('Coordenada X')
    ax.set_ylabel('Coordenada Y')

    # Salvar o plot no arquivo e também no buffer
    buffer_imagem = io.BytesIO()
    fig.savefig(caminho_completo, format='PNG', dpi=dpi, bbox_inches='tight')
    fig.savefig(buffer_imagem, format='PNG', dpi=dpi, bbox_inches='tight')
    buffer_imagem.seek(0)

    # Fechar a figura para liberar memória
    plt.close(fig)

    return caminho_completo, buffer_imagem

# Aplica a funcao
caminho_arquivo, buffer_imagem = plotar_planta_situacao(pesquisa_sql, valor_sql_limpo)
print(f"Imagem salva em: {caminho_arquivo}")

# Exemplo de uso do buffer:
with open('imagem_lote_teste_buffer.png', 'wb') as f:
    f.write(buffer_imagem.getvalue())

C:\Users\guici\AppData\Local\Temp\ipykernel_28856\1233128899.py:62: UserWarning: Legend does not support handles for PatchCollection instances.
See: https://matplotlib.org/stable/tutorials/intermediate/legend_guide.html#implementing-a-custom-legend-handler
  ax.legend()
C:\Users\guici\AppData\Local\Temp\ipykernel_28856\1233128899.py:62: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend()


Imagem salva em: D:\Google Drive\Meu Drive\Codigos bee6\bee6_arquivos\solo_inteligente\Imagens Planta\imagem_lote_1201610005.png


### Monta um PDF bonito e estruturado

In [44]:
def generate_relatorio_pdf(relatorios_texto, buffer_imagem, pesquisa_sql):
    """
    Gera um PDF contendo:
      - Título: "Relatório Solo Inteligente"
      - Texto (contido em 'relatorios_texto')
      - Imagem (contida em 'buffer_imagem')
      
    O arquivo é salvo em:
    D:\\Drive\\Meu Drive\\Codigos bee6\\bee6_arquivos\\solo_inteligente\\Pdfs Prontos

    e o nome do arquivo será:
    relatorio_{valor_sql}.pdf
    """

    # 3) Montar caminho de saída com um nome de arquivo válido
    pasta_saida = r"D:\Google Drive\Meu Drive\Codigos bee6\bee6_arquivos\solo_inteligente\Pdfs Prontos"
    nome_arquivo = f"relatorio_{valor_sql_limpo}.pdf"
    output_pdf_path = os.path.join(pasta_saida, nome_arquivo)

    # 4) Configurar PDF
    doc = SimpleDocTemplate(output_pdf_path, pagesize=A4)
    elements = []

    # 5) Estilos
    styles = getSampleStyleSheet()
    title_style = ParagraphStyle(
        "Title",
        parent=styles["Heading1"],
        fontName="Helvetica-Bold",  # Se quiser usar fonte customizada, registre aqui (Montserrat etc.)
        fontSize=18,
        textColor=colors.HexColor("#333333"),
        alignment=1)
    normal_style = ParagraphStyle(
        "Normal",
        parent=styles["Normal"],
        fontName="Helvetica",
        fontSize=12,
        leading=14,)

    # Título
    elements.append(Paragraph("Relatório Solo Inteligente", title_style))
    elements.append(Spacer(1, 24))

    # Quebra o texto em parágrafos
    paragraphs = relatorios_texto.split("\n\n")
    for paragraph in paragraphs:
        elements.append(Paragraph(paragraph, normal_style))
        elements.append(Spacer(1, 12))

    # 6) Tratar a imagem
    img = PILImage.open(buffer_imagem)
    img_width, img_height = img.size

    max_width, max_height = A4[0] - 100, A4[1] / 2
    aspect_ratio = img_width / img_height

    if img_width > max_width or img_height > max_height:
        if aspect_ratio > 1:  # Largura maior que altura
            pdf_width = max_width
            pdf_height = max_width / aspect_ratio
        else:  # Altura maior que largura
            pdf_height = max_height
            pdf_width = max_height * aspect_ratio
    else:
        pdf_width, pdf_height = img_width, img_height

    img_buffer = BytesIO()
    img.save(img_buffer, format="PNG")
    img_buffer.seek(0)

    image = Image(img_buffer, width=pdf_width, height=pdf_height)
    elements.append(image)

    # 7) Construir e salvar PDF
    doc.build(elements)
    print(f"PDF gerado em: {output_pdf_path}")

# Gera o PDF
generate_relatorio_pdf(string_relatorio, buffer_imagem, pesquisa_sql)

PDF gerado em: D:\Google Drive\Meu Drive\Codigos bee6\bee6_arquivos\solo_inteligente\Pdfs Prontos\relatorio_1201610005.pdf
